# 第06章 C组人工复核

## 6.3 DeepONet与FNO


（1）日期：2026-11-10；组别：G13。

（2）已上传证据：`A2-G15-正文 (1).md` 和 `B2组-修正清单.md`。

（3）B2相关条目：`CH06-G09-B002`、`CH06-G09-B009`、`CH06-G09-B010`。

（4）复核边界：本 Notebook 是 C组独立数学与数值检查，不是 A2 15000 步 DeepONet 的完整重训，也没有访问未上传的 `A2组-实验.ipynb`。下列输出均为本 Notebook 实际执行所得。

（5）运行环境：Python 3、NumPy；CPU即可。无网络、无需原始图片或训练权重。随机种子6303及后续种子用于本组核验，不能与A2的seed=0结果直接逐位比较。

### 6.3-V1 独立解析解与物理约束

对
$$
-u''(x)=f(x),\quad u(0)=u(1)=0,\quad
f(x)=\sum_{k=1}^{8}\left[a_k\cos(2\pi kx)+b_k\sin(2\pi kx)\right]
$$
逐模积分，并利用端点条件消去余下的一次函数，得到
$$
u_{\mathrm{ref}}(x)=\sum_{k=1}^{8}
\frac{a_k[\cos(2\pi kx)-1]+b_k\sin(2\pi kx)}{(2\pi k)^2}.
$$
这里的常数修正项 $-a_k/(2\pi k)^2$ 不可遗漏，否则一般不满足零边界。用独立中心差分近似二阶导数，检验残差与边界。

In [1]:
import numpy as np
import platform
print("Python:", platform.python_version(), "NumPy:", np.__version__)
rng = np.random.default_rng(6303)
modes = np.arange(1, 9, dtype=float)
a = rng.standard_normal(8) * modes**(-1.5)
b = rng.standard_normal(8) * modes**(-1.5)

def source(x, aa=a, bb=b):
    x = np.asarray(x, dtype=float)
    z = 2.0 * np.pi * modes[:, None] * x[None, :]
    return (aa[:, None] * np.cos(z) + bb[:, None] * np.sin(z)).sum(axis=0)

def exact_solution(x, aa=a, bb=b):
    x = np.asarray(x, dtype=float)
    z = 2.0 * np.pi * modes[:, None] * x[None, :]
    return ((aa[:, None] * (np.cos(z)-1.0) + bb[:, None] * np.sin(z))
            / (2.0 * np.pi * modes[:, None])**2).sum(axis=0)

bound = np.max(np.abs(exact_solution(np.array([0.0, 1.0]))))
x = np.linspace(0.03, 0.97, 251)
h = 1e-4
minus_uxx = -(exact_solution(x+h) - 2*exact_solution(x) + exact_solution(x-h)) / h**2
res = np.max(np.abs(minus_uxx - source(x)))
print("V1 boundary max =", f"{bound:.3e}",
      "central-difference residual max =", f"{res:.3e}")
assert bound < 1e-12
assert res < 1e-4

Python: 3.13.5 NumPy: 2.3.5
V1 boundary max = 2.408e-21 central-difference residual max = 3.533e-07


### 6.3-V2 传统数值基线：Thomas法求有限差分，不复用A2的逆矩阵代码

使用内部节点 $x_i=i/(M+1)$、$h=1/(M+1)$ 以及
$$
(-U_{i-1}+2U_i-U_{i+1})/h^2=f(x_i),\qquad U_0=U_{M+1}=0.
$$
逐一采用 $M=32,64,128,256,512$，与V1解析解比较同一节点上的相对 $L^2$ 误差。由于这几档的 $h$ 并非精确减半，观测阶应按实际 $h$ 比值取对数。

In [2]:
def fd_thomas(M, aa=a, bb=b):
    h = 1.0/(M+1)
    nodes = np.arange(1, M+1)*h
    rhs = source(nodes, aa, bb).copy()
    diag = np.full(M, 2.0/h**2)
    off = -1.0/h**2
    for i in range(1, M):
        factor = off/diag[i-1]
        diag[i] -= factor*off
        rhs[i] -= factor*rhs[i-1]
    U = np.empty(M)
    U[-1] = rhs[-1]/diag[-1]
    for i in range(M-2, -1, -1):
        U[i] = (rhs[i]-off*U[i+1])/diag[i]
    return nodes, U

hs, errs = [], []
for M in (32, 64, 128, 256, 512):
    nodes, Ufd = fd_thomas(M)
    truth = exact_solution(nodes)
    err = np.linalg.norm(Ufd-truth)/np.linalg.norm(truth)
    hs.append(1.0/(M+1))
    errs.append(err)
    order = (np.log(errs[-2]/err)/np.log(hs[-2]/hs[-1])) if len(errs)>1 else None
    print(f"M={M:3d}, h={hs[-1]:.7f}, relL2={err:.8e}, "
          f"order={'—' if order is None else f'{order:.4f}'}")
assert all(1.9 < np.log(errs[i-1]/errs[i])/np.log(hs[i-1]/hs[i]) < 2.1
           for i in range(1, len(errs)))

M= 32, h=0.0303030, relL2=4.03737351e-03, order=—
M= 64, h=0.0153846, relL2=1.03878716e-03, order=2.0026
M=128, h=0.0077519, relL2=2.63606722e-04, order=2.0007
M=256, h=0.0038911, relL2=6.64069156e-05, order=2.0002
M=512, h=0.0019493, relL2=1.66659682e-05, order=2.0000


### 6.3-V3 DeepONet：维度与算子泛化的划分单位

（1）输入传感器：$A\in\mathbb R^{N\times S}$；

（2）branch 输出：$B\in\mathbb R^{N\times p}$；

（3）trunk 输出：$T\in\mathbb R^{Q\times p}$；

（4）预测：$\widehat U=BT^\top\in\mathbb R^{N\times Q}$。

下面以随机数组验证矩阵乘法与逐样本内积一致；函数ID拆分只是数据组织最小示例，不是A2训练复现。若只按网格点随机拆分同一函数，训练和测试依旧共享同一个函数ID。

In [3]:
dim_rng = np.random.default_rng(6304)
N,S,Q,p = 7,51,63,32
A = dim_rng.normal(size=(N,S))
B = dim_rng.normal(size=(N,p))
T = dim_rng.normal(size=(Q,p))
Y = B @ T.T
by_loop = np.array([[sum(B[n,k]*T[q,k] for k in range(p))
                     for q in range(Q)] for n in range(N)])
assert A.shape == (7,51) and Y.shape == (7,63)
assert np.allclose(Y, by_loop, atol=1e-12)
train_functions = set(range(800))
test_functions = set(range(800,1000))
assert train_functions.isdisjoint(test_functions)
point_split_same_function = {"train": {5}, "test": {5}}
assert not point_split_same_function["train"].isdisjoint(point_split_same_function["test"])
print("V3 shapes:", A.shape, B.shape, T.shape, Y.shape)
print("whole-function split overlap:", len(train_functions & test_functions),
      "| point-wise split function overlap:",
      len(point_split_same_function["train"] & point_split_same_function["test"]))

V3 shapes: (7, 51) (7, 32) (63, 32) (7, 63)
whole-function split overlap: 0 | point-wise split function overlap: 1


### 6.3-V4 梯度/指标：A2实际训练并不是逐函数相对 $L^2$

A2代码（正文L972）：
```python
D = (Y-U) / (U.shape[0]*U.shape[1])
```
这是 $J(Y)=\tfrac1{2NQ}\|Y-U\|_F^2$ 对 $Y$ 的精确导数，不是
$\frac1N\sum_n\|Y_n-U_n\|_2/\|U_n\|_2$ 的导数。
A2代码（L991）的训练误差 $\|Y-U\|_F/\|U\|_F$ 亦不是逐函数误差的最大值。

In [4]:
U = np.array([[1.0,0.0],[100.0,0.0]])
Y = np.array([[1.1,0.0],[101.0,0.0]])
D = (Y-U)/(U.shape[0]*U.shape[1])
def J(pred):
    return 0.5*np.mean((pred-U)**2)
eps = 1e-6
numgrad = np.zeros_like(Y)
for idx in np.ndindex(Y.shape):
    z1, z2 = Y.copy(), Y.copy()
    z1[idx] += eps; z2[idx] -= eps
    numgrad[idx] = (J(z1)-J(z2))/(2*eps)
assert np.allclose(D, numgrad, atol=1e-10)
per_function = np.linalg.norm(Y-U,axis=1)/np.linalg.norm(U,axis=1)
global_rel = np.linalg.norm(Y-U)/np.linalg.norm(U)
print("V4 D matches gradient of 1/2 MSE:", np.max(abs(D-numgrad)))
print("per-function relL2:", per_function.tolist(),
      "| mean:", per_function.mean(),
      "| max:", per_function.max(),
      "| global relL2:", global_rel)
assert abs(global_rel-per_function.max()) > 1e-2

V4 D matches gradient of 1/2 MSE: 6.311893230304122e-10
per-function relL2: [0.10000000000000009, 0.01] | mean: 0.05500000000000004 | max: 0.10000000000000009 | global relL2: 0.010049373165023728


### 6.3-V5 系数分布：$\mathcal N(0,k^{-1.5})$ 与程序的方差矛盾

通常 $\mathcal N(\mu,\sigma^2)$ 中第二参数表示方差。A2正文L901写 $a_k,b_k\sim\mathcal N(0,k^{-1.5})$，而代码L934–L936写 `randn * k**(-1.5)`，使标准差为 $k^{-1.5}$、方差为 $k^{-3}$。

下方用直接方差公式与大样本数值检查对应关系。本组没有用这些随机样本替代A2的训练数据。

In [5]:
k = np.arange(1,9,dtype=float)
std_code = k**(-1.5)
variance_code = std_code**2
variance_claim = k**(-1.5)
v_rng = np.random.default_rng(6305)
draws = v_rng.standard_normal(size=(50000,8))*std_code
emp = np.var(draws,axis=0)
print("V5 k | stated variance | code variance | empirical variance")
for j in (1,7):
    print(f"  {j+1} | {variance_claim[j]:.8f} | "
          f"{variance_code[j]:.8f} | {emp[j]:.8f}")
assert np.allclose(emp,variance_code,rtol=0.04,atol=0)
assert np.any(np.abs(variance_code[1:]-variance_claim[1:])>1e-3)

V5 k | stated variance | code variance | empirical variance
  2 | 0.35355339 | 0.12500000 | 0.12470461
  8 | 0.04419417 | 0.00195313 | 0.00195464


### 6.3-V6 FNO：一维输入通道与RFFT形状

对于仅有函数值 $f(x)$ 和坐标 $x$ 的一维稳态问题，输入为 `[N,Q,2]`，不是正文L770的 `[N,S,3]`。当沿网格长度 $Q$ 的轴使用 `rfft`，谱长度总是 $\lfloor Q/2\rfloor+1$。对 $Q$ 为奇数和偶数分别验证。此处只复核张量/傅里叶算术，没有训练FNO。

In [6]:
fft_rng = np.random.default_rng(6306)
for Q in (255,256):
    f = fft_rng.normal(size=(2,Q,1))
    xx = np.broadcast_to(np.linspace(0,1,Q)[None,:,None],(2,Q,1))
    inp = np.concatenate([f,xx],axis=2)     # N,Q,2
    hidden = fft_rng.normal(size=(2,8,Q))  # N,w,Q
    F = np.fft.rfft(hidden,axis=-1)
    rec = np.fft.irfft(F,n=Q,axis=-1)
    assert inp.shape == (2,Q,2)
    assert F.shape == (2,8,Q//2+1)
    assert np.allclose(rec,hidden,atol=1e-12)
    print("V6 Q=",Q," input=",inp.shape," spectral=",F.shape)
toy = np.array([1.0,0.0,-1.0,0.0])
assert np.allclose(np.fft.fft(toy),[0,2,0,2])
print("DFT [1,0,-1,0] =",np.fft.fft(toy))

V6 Q= 255  input= (2, 255, 2)  spectral= (2, 8, 128)
V6 Q= 256  input= (2, 256, 2)  spectral= (2, 8, 129)
DFT [1,0,-1,0] = [0.+0.j 2.+0.j 0.+0.j 2.+0.j]


### 6.3-V7 成本图：统一单位后比较

A2正文表6.3-5记录：数据生成1000函数共0.347 s、训练73.8 s、200函数推理2.05 ms。代码L1021–L1031将训练73.8秒直接与其他项的毫秒值放在一个数组里。

该检查只验证单位算术与图示口径，不声称这些时间经C组重新计时，也不把“数据生成耗时（包含逆矩阵和批量生成）”视作严格公平的在线FD求解器基线。

In [7]:
A2_gen_s, A2_train_s, A2_infer_ms = 0.347, 73.8, 2.05
gen_per_fn_ms = A2_gen_s/1000*1000
train_once_ms = A2_train_s*1000
infer_per_fn_ms = A2_infer_ms/200
print("V7 normalized units:")
print(f"  data generation = {gen_per_fn_ms:.5f} ms/function")
print(f"  training        = {train_once_ms:.1f} ms (one-time)")
print(f"  inference       = {infer_per_fn_ms:.5f} ms/function")
assert train_once_ms == 73800.0
assert np.isclose(infer_per_fn_ms,0.01025)

V7 normalized units:
  data generation = 0.34700 ms/function
  training        = 73800.0 ms (one-time)
  inference       = 0.01025 ms/function


### 6.3-V8 标签误差并非一般意义的“模型精度绝对上限”

若教师标签 $\widetilde u=u_{\mathrm{ref}}+\delta$，一个额外利用准确物理知识的模型原则上可以输出 $u_{\mathrm{ref}}$，其真值误差为0，优于标签自身误差。严格地仅有三角不等式
$$
\|\widehat u-u_{\mathrm{ref}}\|\le\|\widehat u-\widetilde u\|+\|\widetilde u-u_{\mathrm{ref}}\|.
$$
A2“标签多准，模型最多就多准”可作为提醒标签质量的经验性说法，不应写成无条件数学定理。本例仅构成逻辑反例，未训练能够纠偏的实际模型。

In [8]:
grid = np.linspace(0,1,101)
ground_truth = np.sin(np.pi*grid)
biased_label = ground_truth+0.1
hypothetical_physically_correct_prediction = ground_truth.copy()
label_error = np.linalg.norm(biased_label-ground_truth)
prediction_error = np.linalg.norm(hypothetical_physically_correct_prediction-ground_truth)
print("V8 label absolute L2 error =",f"{label_error:.5f}",
      "| hypothetical prediction error =",f"{prediction_error:.1f}")
assert label_error>0 and prediction_error==0

V8 label absolute L2 error = 1.00499 | hypothetical prediction error = 0.0


### 6.3-V9 右端项与标签错配的最小反例

A2正文L1082讲述曾误将不同系数的 $f$ 与 $u$ 配对，导致训练失败。C组无法核验该历史运行的“0.94”指标，但可以独立证明：对于 $u(f_2)$，计算 $-u''-f_1=f_2-f_1$，除 $f_1=f_2$ 外残差一般不为0。

In [9]:
pair_rng = np.random.default_rng(6307)
a2 = pair_rng.standard_normal(8)*modes**(-1.5)
b2 = pair_rng.standard_normal(8)*modes**(-1.5)
nodes = np.linspace(0.01,0.99,201)
matched_analytic_res = source(nodes,a2,b2)-source(nodes,a2,b2)
mismatched_analytic_res = source(nodes,a2,b2)-source(nodes,a,b)
print("V9 matched residual RMS =",np.sqrt(np.mean(matched_analytic_res**2)),
      "| mismatched residual RMS =",np.sqrt(np.mean(mismatched_analytic_res**2)))
assert np.sqrt(np.mean(mismatched_analytic_res**2))>0.1

V9 matched residual RMS = 0.0 | mismatched residual RMS = 1.6081840053976877


### 复核结果与未覆盖范围

| 编号 | 独立检查 | 判断 |
| --- | --- | --- |
| V1 | 解族的零边界与数值二阶残差 | 已运行，通过 |
| V2 | FD节点误差二阶收敛（新系数样本） | 已运行，通过 |
| V3 | DeepONet矩阵形状、拆函数与拆点含义 | 已运行，通过 |
| V4 | A2所列 `D` 为半MSE梯度；全局误差不等于逐函数最大值 | 已运行，通过 |
| V5 | A2正态分布参数与代码实际方差不符 | 已运行，确认 |
| V6 | FNO一维两通道与RFFT维数 | 已运行，确认 |
| V7 | A2成本图秒/毫秒混合 | 已计算，确认 |
| V8 | “标签精度为模型绝对上限”的反例 | 教学构造，逻辑有效 |
| V9 | 输入-标签错配使方程不相容 | 已运行，通过 |

未覆盖：A2 15000步 DeepONet 完整重训、原始训练Notebook保存输出对照、A1正文与B1报告、FNO学习效果、硬件计时可比性。B2关于旧输出不一致的报告应保留为“B2报告，C未重跑”，不得写成C组已独立复现。